# Ch 14 · Sensors, Delays and Inverse Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/14-sensors-delays-inverse-dynamics.ipynb)

Companion notebook for the chapter page `chapters/14-sensors-delays-inverse-dynamics.html` ([page](../chapters/14-sensors-delays-inverse-dynamics.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

**How this notebook works.** Every example cell is the same file as on the chapter page (`examples/14-sensors-delays-inverse-dynamics/`). Budgets follow `ON_GPU`. The next cell writes the running helper module `my_env.py` (snapshot after Chapter 14: everything from Ch 6-10 plus the new `sensor_slices`), so the notebook runs in an empty `/content`.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 14.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 added randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Ch 14 adds sensor_slices (Example 14.3): sensor name -> (adr, dim) in d.sensordata.
Later chapters extend this file (Ch 24 capture_step, Ch 26 obs/reward kernels,
Ch 30 the environments).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize", "sensor_slices"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


# --- Ch 14: sensors ------------------------------------------------------------------------------
def sensor_slices(mjm):
    """Return {sensor name: (adr, dim)} so d.sensordata[:, adr:adr + dim] is that sensor.

    Unnamed sensors get the key "sensor<i>". The addresses come from the compiled MjModel
    (mjm.sensor_adr, mjm.sensor_dim) and are the same in every world.
    """
    out = {}
    for i in range(mjm.nsensor):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i) or f"sensor{i}"
        out[name] = (int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i]))
    return out

## 1. Sensor stages and the catalogue
`forward` zeroes `d.sensordata`, then calls `sensor_pos` after `fwd_position`, `sensor_vel` after `fwd_velocity` and `sensor_acc` after the solver. Example 14.1 builds a model with one sensor of every supported type and prints stage, address and dim (the full enum is the private `mujoco_warp._src.types.SensorType`).

In [ ]:
# Example 14.1 - ex1_sensor_catalogue.py
"""Example 14.1 - the sensor catalogue: every SensorType, its stage and its dim.

Builds one MJCF that contains one sensor of every kind MJWarp 4.0.1 supports, compiles it with
MuJoCo, uploads it with put_model, runs one forward pass and prints, for each sensor: name, type,
the stage that computes it (mjm.sensor_needstage), its slice of sensordata (adr, dim), and which
put_model list it was sorted into. The only mjtSensor value missing from SensorType is PLUGIN.
Run: python ex1_sensor_catalogue.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.types import SensorType   # private API: full enum (not exported by mjw)

np.set_printoptions(precision=4, suppress=True)

CATALOGUE = """
<mujoco model="catalogue">
  <option timestep="0.005" magnetic="0 -0.5 0"/>
  <asset><mesh name="pad" builtin="sphere" params="0" scale=".05 .05 .05"/></asset>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 .1"/>
    <camera name="cam" pos="0 -1.5 .5" xyaxes="1 0 0 0 0 1" resolution="64 48"/>
    <site name="zone" pos="0 0 .3" type="box" size=".5 .5 .5"/>
    <body name="arm" pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0" range="-1 1"/>
      <geom name="link" type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass="1"/>
      <site name="imu" pos=".3 0 0"/>
      <body name="wrist" pos=".3 0 0">
        <joint name="ball" type="ball"/>
        <geom name="hand" type="sphere" size=".04" mass=".2"/>
      </body>
    </body>
    <body name="box" pos="1 0 .1">
      <freejoint/>
      <geom name="boxgeom" type="box" size=".1 .1 .1" mass="1"/>
      <site name="boxsite" pos="0 0 -.1"/>
      <site name="ray" pos="0 0 .1" zaxis="0 0 1"/>
    </body>
    <body name="padbody" pos="-1 0 .5">
      <freejoint/>
      <geom name="padgeom" type="mesh" mesh="pad"/>
    </body>
  </worldbody>
  <tendon><fixed name="ten" range="-1 1"><joint joint="hinge" coef="1"/></fixed></tendon>
  <actuator>
    <motor name="m_hinge" joint="hinge" gear="1"/>
    <motor name="m_ten" tendon="ten" gear="1"/>
  </actuator>
  <sensor>
    <magnetometer name="magnetometer" site="imu"/>
    <camprojection name="camprojection" site="imu" camera="cam"/>
    <rangefinder name="rangefinder" site="ray"/>
    <jointpos name="jointpos" joint="hinge"/>
    <tendonpos name="tendonpos" tendon="ten"/>
    <actuatorpos name="actuatorpos" actuator="m_hinge"/>
    <ballquat name="ballquat" joint="ball"/>
    <jointlimitpos name="jointlimitpos" joint="hinge"/>
    <tendonlimitpos name="tendonlimitpos" tendon="ten"/>
    <framepos name="framepos" objtype="site" objname="imu"/>
    <framexaxis name="framexaxis" objtype="site" objname="imu"/>
    <frameyaxis name="frameyaxis" objtype="site" objname="imu"/>
    <framezaxis name="framezaxis" objtype="site" objname="imu"/>
    <framequat name="framequat" objtype="site" objname="imu"/>
    <subtreecom name="subtreecom" body="arm"/>
    <distance name="geomdist" geom1="boxgeom" geom2="floor"/>
    <normal name="geomnormal" geom1="boxgeom" geom2="floor"/>
    <fromto name="geomfromto" geom1="boxgeom" geom2="floor"/>
    <insidesite name="insidesite" site="zone" objtype="body" objname="box"/>
    <e_potential name="e_potential"/>
    <e_kinetic name="e_kinetic"/>
    <clock name="clock"/>
    <velocimeter name="velocimeter" site="imu"/>
    <gyro name="gyro" site="imu"/>
    <jointvel name="jointvel" joint="hinge"/>
    <tendonvel name="tendonvel" tendon="ten"/>
    <actuatorvel name="actuatorvel" actuator="m_hinge"/>
    <ballangvel name="ballangvel" joint="ball"/>
    <jointlimitvel name="jointlimitvel" joint="hinge"/>
    <tendonlimitvel name="tendonlimitvel" tendon="ten"/>
    <framelinvel name="framelinvel" objtype="site" objname="imu"/>
    <frameangvel name="frameangvel" objtype="site" objname="imu"/>
    <subtreelinvel name="subtreelinvel" body="arm"/>
    <subtreeangmom name="subtreeangmom" body="arm"/>
    <touch name="touch" site="boxsite"/>
    <contact name="contact" body1="box" num="2" data="found force"/>
    <accelerometer name="accelerometer" site="imu"/>
    <force name="force" site="imu"/>
    <torque name="torque" site="imu"/>
    <actuatorfrc name="actuatorfrc" actuator="m_hinge"/>
    <tendonactuatorfrc name="tendonactfrc" tendon="ten"/>
    <jointactuatorfrc name="jointactfrc" joint="hinge"/>
    <jointlimitfrc name="jointlimitfrc" joint="hinge"/>
    <tendonlimitfrc name="tendonlimitfrc" tendon="ten"/>
    <framelinacc name="framelinacc" objtype="site" objname="imu"/>
    <frameangacc name="frameangacc" objtype="site" objname="imu"/>
    <tactile name="tactile" geom="padgeom" mesh="pad"/>
    <user name="user" dim="2" needstage="acc" datatype="real"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2

mjm = mujoco.MjModel.from_xml_string(CATALOGUE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
mjw.forward(m, d)
wp.synchronize()

STAGE = {int(mujoco.mjtStage.mjSTAGE_POS): "pos", int(mujoco.mjtStage.mjSTAGE_VEL): "vel",
         int(mujoco.mjtStage.mjSTAGE_ACC): "acc"}
# the per-stage launch lists that put_model builds (io.py)
lists = {name: set(getattr(m, name).numpy().tolist()) for name in (
    "sensor_pos_adr", "sensor_limitpos_adr", "sensor_vel_adr", "sensor_limitvel_adr",
    "sensor_acc_adr", "sensor_touch_adr", "sensor_limitfrc_adr", "sensor_tendonactfrc_adr",
    "sensor_contact_pos_adr", "sensor_contact_vel_adr", "sensor_contact_acc_adr")}

rows = []
seen = set()
print(f"{'sensor':15s} {'SensorType':15s} stage adr dim  launched by")
for i in range(mjm.nsensor):
    st = SensorType(int(mjm.sensor_type[i]))
    seen.add(st)
    stage = STAGE[int(mjm.sensor_needstage[i])]
    adr, dim = int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i])
    where = [k.replace("sensor_", "").replace("_adr", "") for k, v in lists.items() if i in v]
    # the generic pos/vel/acc lists also hold RANGEFINDER, TACTILE and USER; their kernels have no
    # branch for these types (the thread writes nothing), the function named after it fills the columns
    if st == SensorType.RANGEFINDER:
        where = where + ["rangefinder (ray.rays)"]
    elif st in (SensorType.E_POTENTIAL, SensorType.E_KINETIC):
        where = where + ["energy_pos/vel"]
    elif st == SensorType.TACTILE:
        where = where + ["_sensor_tactile"]
    elif st == SensorType.USER:
        where = where + ["callback.sensor"]
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i)
    print(f"{name:15s} {st.name:15s} {stage:5s} {adr:3d} {dim:3d}  {', '.join(where)}")
    rows.append({"name": name, "type": st.name, "stage": stage, "adr": adr, "dim": dim})

missing = [s.name for s in SensorType if s not in seen]
print("nsensor:", mjm.nsensor, "| nsensordata:", mjm.nsensordata,
      "| sensordata shape:", d.sensordata.shape)
print("SensorType members covered:", len(seen), "of", len(SensorType), "| missing:", missing)
not_supported = [e.name for e in mujoco.mjtSensor.__members__.values()
                 if int(e) not in {int(s) for s in SensorType}]
print("mjtSensor values put_model rejects:", not_supported)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"sensors": rows, "nsensordata": int(mjm.nsensordata)}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json, len(rows), "sensors")

## 2. An IMU: accelerometer, gyro, magnetometer
The accelerometer measures specific force (acceleration minus gravity): +9.81 along 'up' at rest, 0 in free fall. The magnetometer is `site_xmat^T @ opt.magnetic`.

In [ ]:
# Example 14.2 - ex2_imu_magnetometer.py
"""Example 14.2 - an IMU: accelerometer, gyro and magnetometer on a box.

World 0: the box rests flat on the floor. World 1: rests rotated 90 deg about x (on its side).
World 2: tumbles in free fall (high up, spinning). After 200 steps we print each sensor and check
  * accelerometer = specific force: +9.81 along the site's "up" axis at rest, ~0 in free fall;
  * gyro = angular velocity in the site frame (0 at rest);
  * magnetometer = site_xmat^T @ opt.magnetic (sensor.py::_magnetometer).
Run: python ex2_imu_magnetometer.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

IMU_BOX = """
<mujoco model="imu_box">
  <option timestep="0.005" magnetic="0 -0.5 0"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box" pos="0 0 .1">
      <freejoint/>
      <geom type="box" size=".1 .1 .1" mass="1"/>
      <site name="imu"/>
    </body>
  </worldbody>
  <sensor>
    <accelerometer name="acc" site="imu"/>
    <gyro name="gyro" site="imu"/>
    <magnetometer name="mag" site="imu"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 3            # three hand-made worlds: same on CPU and GPU
NSTEP = 200

mjm = mujoco.MjModel.from_xml_string(IMU_BOX)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
c = np.cos(np.pi / 4)
qpos[1, 3:7] = [c, c, 0, 0]          # world 1: rotated 90 deg about x, resting on a side face
qpos[2, 2] = 50.0                    # world 2: high in the air ...
qvel[2, 3:6] = [2.0, 0.0, 5.0]       # ... and spinning (body-frame angular velocity, rad/s)
d.qpos.assign(qpos)
d.qvel.assign(qvel)

for _ in range(NSTEP):
    mjw.step(m, d)
mjw.forward(m, d)       # sensors of the final state
wp.synchronize()

sd = d.sensordata.numpy()
acc, gyro, mag = sd[:, 0:3], sd[:, 3:6], sd[:, 6:9]
xmat = d.site_xmat.numpy()[:, 0]                       # (nworld, 3, 3)
magnetic = mjm.opt.magnetic
labels = ["rest flat", "rest on side", "free fall + spin"]
for w in range(NWORLD):
    print(f"world {w} ({labels[w]}):")
    print("  accelerometer", acc[w], "| norm", f"{np.linalg.norm(acc[w]):.3f}")
    print("  gyro         ", gyro[w])
    print("  magnetometer ", mag[w], "| R^T @ magnetic", xmat[w].T @ magnetic)
err = max(np.abs(mag[w] - xmat[w].T @ magnetic).max() for w in range(NWORLD))
print(f"max |magnetometer - R^T magnetic| = {err:.2e}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())


def r3(a):
    return np.round(np.asarray(a, dtype=float), 3).tolist()     # float64 first: clean JSON


if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"labels": labels, "acc": r3(acc), "gyro": r3(gyro), "mag": r3(mag),
                   "magnetic": magnetic.tolist()}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 3. Reading sensordata by address
`d.sensordata` has shape `(nworld, nsensordata)`. Sensor i lives in columns `sensor_adr[i] : sensor_adr[i] + sensor_dim[i]`. `my_env.sensor_slices(mjm)` builds the name map.

In [ ]:
# Example 14.3 - ex3_sensordata_slices.py
"""Example 14.3 - reading d.sensordata by address: one row per world, one slice per sensor.

PENDULUM (M2) with a different constant shoulder torque in every world. After 100 steps we slice
d.sensordata[:, adr:adr + dim] for each sensor using my_env.sensor_slices(mjm) (new in this
chapter) and print the per-world matrix. The model enables the energy flag, so d.energy is filled too.
Run: python ex3_sensordata_slices.py [--dump-json PATH]
"""
import argparse
import json
import pathlib
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

HERE = pathlib.Path(__file__).parent if "__file__" in globals() else pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
try:
    import my_env                           # notebook: written by the %%writefile cell
except ImportError:
    import my_env_step14 as my_env          # examples/14-sensors-delays-inverse-dynamics/

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
d.ctrl.assign(np.linspace(-2.0, 2.0, NWORLD, dtype=np.float32)[:, None])   # torque per world

for _ in range(NSTEP):
    mjw.step(m, d)
mjw.forward(m, d)          # sensors for the final state (step computed them before integrating)
wp.synchronize()

slices = my_env.sensor_slices(mjm)
print("sensor_slices:", slices)
print("sensordata shape:", d.sensordata.shape, "(nworld, nsensordata)")
sd = d.sensordata.numpy()
show = np.linspace(0, NWORLD - 1, min(NWORLD, 4)).astype(int)   # 4 evenly spaced worlds (0..3 on CPU)
print("worlds shown:", show)
print("ctrl per world:", d.ctrl.numpy()[show, 0])
for name, (adr, dim) in slices.items():
    print(f"d.sensordata[:, {adr}:{adr + dim}]  # {name}")
    print(sd[show, adr:adr + dim])
# the sensor slices agree with the state they measure
q1_adr = slices["q1"][0]
err_q = np.abs(sd[:, q1_adr] - d.qpos.numpy()[:, 0]).max()
pa, pd = slices["tip_pos"]
err_p = np.abs(sd[:, pa:pa + pd] - d.site_xpos.numpy()[:, 0]).max()
print(f"max |q1 - qpos[:,0]| = {err_q:.1e}, max |tip_pos - site_xpos| = {err_p:.1e}")
# PENDULUM enables <flag energy="enable"/>: forward also fills d.energy = [potential, kinetic]
print("d.energy:", d.energy.shape, "array of wp.vec2 | worlds shown [E_potential, E_kinetic]:")
print(d.energy.numpy()[show])

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"slices": slices, "ctrl": np.round(d.ctrl.numpy()[show, 0].astype(float), 3).tolist(),
                   "sensordata": np.round(sd[show].astype(float), 3).tolist()}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 4. Rays and collisions as sensors
Rangefinder outputs are chosen with `data="..."`; the compiler stores a `RayDataField` bitmask in `sensor_intprm[:, 0]`. A camera rangefinder casts one ray per pixel.

In [ ]:
# Example 14.4 - ex4_rangefinder_dataspec.py
"""Example 14.4 - rangefinder outputs chosen with data="...": the RayDataField bitmask.

A downward-looking site rangefinder with data="dist point normal" rides on a slider at a different
height in every world, over a floor with a raised block. A second rangefinder is attached to a tiny
4x3 downward camera (one ray per pixel) with data="dist". The camera sits in its own static body
"mast": rays exclude the geoms of the sensor's own body, so a camera on the worldbody would not see
the floor. We print the bitmask stored in
mjm.sensor_intprm[:, 0], the resulting sensor dims (1 + 3 + 3 = 7 and 4*3*1 = 12), and the readings.
Run: python ex4_rangefinder_dataspec.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

RANGE_SCENE = """
<mujoco model="rangefinder">
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <geom name="block" type="box" pos=".6 0 .1" size=".2 .2 .1"/>
    <body name="probe" pos="0 0 1">
      <joint name="z" type="slide" axis="0 0 1"/>
      <joint name="x" type="slide" axis="1 0 0"/>
      <geom type="sphere" size=".02" contype="0" conaffinity="0"/>
      <site name="down" zaxis="0 0 -1"/>
    </body>
    <body name="mast" pos="0.3 0 1.5">
      <camera name="eye" xyaxes="1 0 0 0 1 0" fovy="60" resolution="4 3"/>
    </body>
  </worldbody>
  <sensor>
    <rangefinder name="laser" site="down" data="dist point normal"/>
    <rangefinder name="depthcam" camera="eye" data="dist"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 4            # four hand-picked probe poses (same on CPU and GPU)

mjm = mujoco.MjModel.from_xml_string(RANGE_SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

# probe pose per world: (dz, dx) relative to the body position (0, 0, 1)
pose = np.array([[0.0, 0.0], [-0.5, 0.0], [0.0, 0.6], [-0.5, 0.6]], dtype=np.float32)
d.qpos.assign(pose)
mjw.forward(m, d)
wp.synchronize()

for i in range(mjm.nsensor):
    mask = int(mjm.sensor_intprm[i, 0])
    fields = [f.name for f in mjw.RayDataField if mask & (1 << f)]
    name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i)
    print(f"{name}: sensor_intprm[{i}, 0] = {mask:#08b} -> {fields}, dim = {mjm.sensor_dim[i]}")
print("rays cast per world (m.nrangefinder):", m.nrangefinder)

sd = d.sensordata.numpy()
for w in range(NWORLD):
    dist, point, normal = sd[w, 0], sd[w, 1:4], sd[w, 4:7]
    probe_z = 1.0 + pose[w, 0]
    print(f"world {w}: probe at x={pose[w, 1]:.1f} z={probe_z:.1f} | dist {dist:.4f} "
          f"point {point} normal {normal}")
cam = sd[:, 7:19].reshape(NWORLD, 3, 4)      # row-major pixels, 1 value (dist) per ray
print("camera rangefinder, world 0, dist per pixel (3 rows x 4 cols):")
print(cam[0])
mjd = mujoco.MjData(mjm)                     # same world 0 in MuJoCo C for comparison
mjd.qpos[:] = pose[0]
mujoco.mj_forward(mjm, mjd)
print("max |MJWarp - MuJoCo C| world 0:", f"{np.abs(sd[0] - mjd.sensordata).max():.1e}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"pose": np.round(pose.astype(float), 3).tolist(),
                   "laser": np.round(sd[:, :7].astype(float), 4).tolist(),
                   "cam": np.round(cam[0].astype(float), 4).tolist()}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

Collision sensors (`distance`, `normal`, `fromto`) run through the collision pipeline.

In [ ]:
# Example 14.5 - ex5_geomdist.py
"""Example 14.5 - collision sensors: distance (GEOMDIST), normal and fromto between two geoms.

A ball (radius 0.1) drops from z=1 onto a floor in every world, from a slightly different height.
A <distance> sensor with cutoff="2" reports the signed ball-floor distance (negative = penetration);
<normal> and <fromto> report the direction and the closest-point segment. These sensors reuse the
collision pipeline (contacts tagged ContactType.SENSOR, sensor.py::_sensor_collision), so the scene
needs contact buffer room. Prints the distance every 50 steps and compares world 0 with MuJoCo C.
Run: python ex5_geomdist.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

DROP = """
<mujoco model="geomdist">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1"/>
    </body>
  </worldbody>
  <sensor>
    <distance name="dist" geom1="ball" geom2="floor" cutoff="2"/>
    <normal name="normal" geom1="ball" geom2="floor" cutoff="2"/>
    <fromto name="fromto" geom1="ball" geom2="floor" cutoff="2"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 300

mjm = mujoco.MjModel.from_xml_string(DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=8, njmax=32)
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)
qpos[:, 2] = np.linspace(1.0, 1.3, NWORLD)
d.qpos.assign(qpos)

mjd = mujoco.MjData(mjm)                     # world 0 in MuJoCo C, for comparison
mjd.qpos[:] = qpos[0]

show = np.linspace(0, NWORLD - 1, min(NWORLD, 4)).astype(int)   # 4 evenly spaced worlds (0..3 on CPU)
frames = []
print("worlds shown:", show)
print("step   t     dist per world (4 shown)             C world 0")
for i in range(NSTEP + 1):
    mjw.forward(m, d)                        # sensors for the current state
    mujoco.mj_forward(mjm, mjd)
    if i % 2 == 0:
        frames.append([round(float(v), 3) for v in d.sensordata.numpy()[show, 0]])
    if i % 50 == 0:
        sd = d.sensordata.numpy()
        print(f"{i:4d} {i * mjm.opt.timestep:5.2f}  {sd[show, 0]}  {mjd.sensordata[0]:.4f}")
        if i == 0:
            normal0, fromto0 = sd[0, 1:4].copy(), sd[0, 4:10].copy()
    if i < NSTEP:
        mjw.step(m, d)
        mujoco.mj_step(mjm, mjd)
wp.synchronize()

sd = d.sensordata.numpy()
print("step 0   (separated)  normal:", normal0, "fromto:", fromto0)
print("step 300 (penetrating) normal:", sd[0, 1:4], "fromto:", sd[0, 4:10])
print("MuJoCo C world 0 normal:", mjd.sensordata[1:4], "fromto:", mjd.sensordata[4:10])
print("ball z (world 0):", f"{d.qpos.numpy()[0, 2]:.4f}", "= radius 0.1 + dist")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"dt": 2 * mjm.opt.timestep, "frames": frames, "nworld": len(show)},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

Touch and contact sensors are acceleration-stage sensors: they read `efc.force`.

In [ ]:
# Example 14.6 - ex6_contact_sensor.py
"""Example 14.6 - touch and contact sensors under a resting box.

A box (mass per world 1, 2, 3, 4 kg) settles on the floor. A <touch> sensor on a site that encloses
the bottom face sums the normal forces in its zone; a <contact> sensor matches contacts of body
"box" and reports up to num=4 of them (reduce="maxforce" keeps the strongest) with
data="found force normal". Both are acceleration-stage sensors (they need efc.force).
Prints the dim layout, the found count, each slot's force and the sum compared with m g.
Run: python ex6_contact_sensor.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BOX = """
<mujoco model="contact_sensor">
  <option timestep="0.005"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="box" pos="0 0 .1">
      <freejoint/>
      <geom name="boxgeom" type="box" size=".1 .1 .1" mass="1"/>
      <site name="sole" pos="0 0 -.1" type="box" size=".11 .11 .02"/>
    </body>
  </worldbody>
  <sensor>
    <touch name="touch" site="sole"/>
    <contact name="contact" body1="box" num="4" reduce="maxforce" data="found force normal"/>
  </sensor>
</mujoco>
"""

NWORLD = 4            # one world per box mass
NSTEP = 200

mjm = mujoco.MjModel.from_xml_string(BOX)
m = mjw.put_model(mjm, batch_sizes={"body_mass": NWORLD})   # per-world mass (Ch 10)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
box = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "box")
bm = m.body_mass.numpy()
bm[:, box] = np.arange(1, NWORLD + 1)
m.body_mass.assign(bm)      # body_inertia stays as for 1 kg: fine for a box resting flat

for _ in range(NSTEP):
    mjw.step(m, d)
mjw.forward(m, d)
wp.synchronize()

print("touch dim:", mjm.sensor_dim[0], "| contact dim:", mjm.sensor_dim[1],
      "= num 4 x (1 found + 3 force + 3 normal)")
sd = d.sensordata.numpy()
g = 9.81
for w in range(NWORLD):
    slots = sd[w, 1:].reshape(4, 7)               # per slot: found(1) force(3) normal(3)
    found = int(slots[0, 0])                      # every slot repeats the match count
    fsum = slots[:found, 1].sum()                 # force[0] is the normal component
    print(f"world {w}: m={bm[w, box]:.0f} kg  m g={bm[w, box] * g:6.3f}  touch={sd[w, 0]:6.3f}  "
          f"found={found}  sum normal force={fsum:6.3f}")
print("world 0 slots [found | f_normal f_t1 f_t2 | nx ny nz]:")
print(sd[0, 1:].reshape(4, 7))
print("contacts per world:", np.bincount(d.contact.worldid.numpy()[:d.nacon.numpy()[0]],
                                         minlength=NWORLD))
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 5. Cutoff and interval sampling
`cutoff` clamps; `interval="period phase"` recomputes a sensor only every period and holds the value in between. Watch the 0.06 s sample: float32 time makes MJWarp sample one step later than C.

In [ ]:
# Example 14.7 - ex7_cutoff_interval.py
"""Example 14.7 - sensor cutoff (clamping) and interval sampling (sample-and-hold).

A 1 kg slider pushed by a constant 1 N force: v = t, x = t^2 / 2, dt = 0.01 s. Four sensors:
  vel_cut   <jointvel cutoff="0.05"/>                     clamped to [-0.05, 0.05]
  pos       <jointpos/>                                   fresh value every step
  pos_int   <jointpos interval="0.03 0" nsample="5"/>       recomputed every 0.03 s, held between
  pos_both  <jointpos delay="0.02" interval="0.03 0" nsample="5"/>  (string from history_test.py)
We print 10 steps of sensordata next to the true state and compare with MuJoCo C at every step.
The one difference (the 0.06 s sample) comes from float32 time; see the last line.
Run: python ex7_cutoff_interval.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

SLIDER = """
<mujoco model="cutoff_interval">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body name="cart">
      <joint name="slide" type="slide" axis="1 0 0"/>
      <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slide" gear="1"/></actuator>
  <sensor>
    <jointvel name="vel_cut" joint="slide" cutoff="0.05"/>
    <jointpos name="pos" joint="slide"/>
    <jointpos name="pos_int" joint="slide" interval="0.03 0" nsample="5"/>
    <jointpos name="pos_both" joint="slide" delay="0.02" interval="0.03 0" nsample="5"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 10

mjm = mujoco.MjModel.from_xml_string(SLIDER)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
d.ctrl.fill_(1.0)
mjd = mujoco.MjData(mjm)
mjd.ctrl[:] = 1.0

print("sensor_cutoff:", mjm.sensor_cutoff, "| sensor_interval (period, phase):",
      mjm.sensor_interval.tolist())
print("nhistory:", mjm.nhistory, "| d.history shape:", d.history.shape)
print(" step  time   qvel   vel_cut     qpos     pos   pos_int  pos_both | C pos_int")
rows, maxerr = [], 0.0
for i in range(NSTEP):
    qvel, qpos, t = d.qvel.numpy()[0, 0], d.qpos.numpy()[0, 0], d.time.numpy()[0]
    mjw.step(m, d)                     # sensors are computed inside step, before integration
    mujoco.mj_step(mjm, mjd)
    sd = d.sensordata.numpy()[0]
    maxerr = max(maxerr, float(np.abs(sd - mjd.sensordata).max()))
    print(f"{i:5d} {t:5.2f} {qvel:6.3f} {sd[0]:8.3f} {qpos:8.5f} {sd[1]:7.5f} {sd[2]:8.5f} "
          f"{sd[3]:8.5f} | {mjd.sensordata[2]:8.5f}")
    rows.append([round(float(t), 3), round(float(qpos), 6)] + [round(float(v), 6) for v in sd])
print(f"max |MJWarp - MuJoCo C| over {NSTEP} steps: {maxerr:.1e}")
# Why step 6 differs: d.time is float32 in MJWarp. Six additions of 0.01 give 0.059999995, and the
# interval test time_prev + period <= t (history.py) is 0.06 <= 0.059999995: False.
t32 = np.float32(0.0)
for _ in range(6):
    t32 = np.float32(t32 + np.float32(0.01))
print("float32 time after 6 steps:", repr(float(t32)), "| time_prev + period:",
      repr(float(np.float32(0.03) + np.float32(0.03))), "| C time:", 6 * 0.01)

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"cols": ["t", "qpos", "vel_cut", "pos", "pos_int", "pos_both"], "rows": rows},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 6. Delays: history ring buffers
An actuator with `delay=0.03` at `dt=0.01` applies ctrl three steps late. The raw buffer is `[user, cursor, times..., values...]`.

In [ ]:
# Example 14.8 - ex8_actuator_delay.py
"""Example 14.8 - actuator delay: <motor delay="0.03" nsample="5"/> at dt = 0.01.

ctrl = 1 from step 0 in every world. The actuator reads ctrl from its history buffer at
time - delay (history.py::read_ctrl_delayed), and _advance inserts the current ctrl just before
time advances (insert_ctrl_history). So actuator_force is 0 for 3 steps, then 1.
We also print the raw ring buffer of world 0 after each step: [user, cursor, times[5], values[5]].
Run: python ex8_actuator_delay.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=True)

DELAYED_MOTOR = """
<mujoco model="actuator_delay">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body name="cart">
      <joint name="slide" type="slide" axis="1 0 0"/>
      <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator><motor joint="slide" delay="0.03" nsample="5"/></actuator>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 6

mjm = mujoco.MjModel.from_xml_string(DELAYED_MOTOR)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
mjd = mujoco.MjData(mjm)

print("actuator_delay:", mjm.actuator_delay, "| actuator_history (nsample, interp):",
      mjm.actuator_history.tolist(), "| nhistory:", mjm.nhistory)
print("history buffer layout: [user, cursor, times x nsample, values x nsample x ctrlnum]")
buf0 = d.history.numpy()[0]
print("initial buffer (world 0):", buf0)

d.ctrl.fill_(1.0)
mjd.ctrl[:] = 1.0
force, force_c, frames = [], [], [{"t": 0.0, "buf": np.round(buf0.astype(float), 3).tolist(), "force": None}]
for i in range(NSTEP):
    t = float(d.time.numpy()[0])
    mjw.step(m, d)
    mujoco.mj_step(mjm, mjd)
    f = float(d.actuator_force.numpy()[0, 0])
    force.append(round(f, 3))
    force_c.append(round(float(mjd.actuator_force[0]), 3))
    buf = d.history.numpy()[0]
    print(f"step {i}: t={t:.2f}  reads ctrl at t-0.03={t - 0.03:+.2f} -> actuator_force {f:.1f} | "
          f"buffer after step: cursor={int(buf[1])} times={buf[2:7]} values={buf[7:12]}")
    frames.append({"t": round(t, 3), "buf": np.round(buf.astype(float), 3).tolist(), "force": round(f, 3)})
print("actuator_force over 6 steps, MJWarp :", force)
print("actuator_force over 6 steps, MuJoCo C:", force_c)
af = d.actuator_force.numpy()
print("all worlds identical:", bool((af == af[0]).all()))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nsample": 5, "dt": 0.01, "delay": 0.03, "frames": frames}, f,
                  separators=(",", ":"))
    print("wrote", args.dump_json)

A delayed sensor lags its fresh copy; `mjw.read_sensor` queries the buffer at any time (it subtracts the sensor's delay).

In [ ]:
# Example 14.9 - ex9_sensor_delay.py
"""Example 14.9 - sensor delay: a delayed jointpos next to a fresh one, and mjw.read_sensor.

A slider coasts at constant speed (world w: qvel = 1 + w, no gravity, no friction), so the fresh
jointpos equals speed * t. The delayed copy (delay="0.02" nsample="4") lags by exactly 2 steps of
dt = 0.01. Then mjw.read_sensor queries the history buffer at chosen times with ZOH (interp=0) and
linear (interp=1) interpolation. Note: read_sensor looks up the buffer at time - delay, i.e. it
returns what the delayed sensor would report at that time (history.py::_read_sensor_kernel).
Run: python ex9_sensor_delay.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

COAST = """
<mujoco model="sensor_delay">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body name="cart">
      <joint name="slide" type="slide" axis="1 0 0"/>
      <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <sensor>
    <jointpos name="fresh" joint="slide"/>
    <jointpos name="delayed" joint="slide" delay="0.02" nsample="4"/>
  </sensor>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 3
NSTEP = 8

mjm = mujoco.MjModel.from_xml_string(COAST)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
d.qvel.assign((1.0 + np.arange(NWORLD, dtype=np.float32))[:, None])

print("sensor_delay:", mjm.sensor_delay, "| sensor_history (nsample, interp):",
      mjm.sensor_history.tolist(), "| nhistory:", mjm.nhistory)
rows = []
print("step  time  fresh(w0) delayed(w0)  fresh(w1) delayed(w1)")
for i in range(NSTEP):
    mjw.step(m, d)            # sensors are evaluated at the state *before* integration
    sd = d.sensordata.numpy()
    t = i * mjm.opt.timestep                   # time at which they were measured
    print(f"{i:4d} {t:5.2f} {sd[0, 0]:9.4f} {sd[0, 1]:11.4f} {sd[1, 0]:10.4f} {sd[1, 1]:11.4f}")
    rows.append([round(t, 3)] + [round(float(v), 4) for v in sd[:2].ravel()])
lag = (sd[:, 0] - sd[:, 1]) / (1.0 + np.arange(NWORLD))
print("lag fresh - delayed, in seconds, per world:", lag[:4], "=", round(lag[0] / 0.01), "steps")

# query the delayed sensor at past times: one query time per world
res = wp.zeros((d.nworld, 1), dtype=float)
qt = np.full(NWORLD, 0.065, dtype=np.float32)             # between two buffer samples
for interp, label in ((0, "ZOH"), (1, "linear")):
    mjw.read_sensor(m, d, 1, wp.array(qt, dtype=float), interp, res)
    print(f"read_sensor(sensor 1, time=0.065, interp={interp} {label}): {res.numpy()[:4, 0]}"
          f"  (buffer at 0.065 - 0.02 = 0.045)")
buf = d.history.numpy()[0]
n = 4
cursor = int(buf[1])
order = [(cursor + 1 + k) % n for k in range(n)]           # _history_physical_index
print("world 0 buffer, oldest -> newest: times", buf[2:2 + n][order], "values",
      buf[2 + n:2 + 2 * n][order])

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"cols": ["t", "fresh_w0", "delayed_w0", "fresh_w1", "delayed_w1"],
                   "rows": rows}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 7. Inverse dynamics
`mjw.inverse` takes `qpos, qvel, qacc` and returns the generalized force `qfrc_inverse` that produces that acceleration. After `forward`, it must equal `qfrc_applied + qfrc_actuator`.

In [ ]:
# Example 14.10 - ex10_inverse_identity.py
"""Example 14.10 - inverse dynamics closes the loop: qfrc_inverse = qfrc_applied + qfrc_actuator.

PLAYGROUND (M3: a ball, a box and a hinged arm with a motor). Every world gets its own random
motor ctrl and random qfrc_applied. After 100 steps (so contacts are active) we run mjw.forward,
which computes qacc, then mjw.inverse, which takes that qacc and recovers the generalized force
that must have produced it:
    qfrc_inverse = M qacc + qfrc_bias - qfrc_passive - qfrc_constraint  (inverse.py::_qfrc_inverse)
Forward dynamics put qfrc_applied + qfrc_actuator (+ J^T xfrc_applied, zero here) in, so the two
must agree up to float32 round-off and the solver tolerance. World 0 is also compared with
mj_inverse.
Run: python ex10_inverse_identity.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
rng = np.random.default_rng(0)
d.ctrl.assign(rng.uniform(-1, 1, (NWORLD, mjm.nu)).astype(np.float32))
d.qfrc_applied.assign(rng.uniform(-0.5, 0.5, (NWORLD, mjm.nv)).astype(np.float32))

for _ in range(NSTEP):
    mjw.step(m, d)
mjw.forward(m, d)            # qacc from the current state and inputs
mjw.inverse(m, d)            # qfrc_inverse from the same state and that qacc
wp.synchronize()

qfrc_in = d.qfrc_applied.numpy() + d.qfrc_actuator.numpy()
err = np.abs(d.qfrc_inverse.numpy() - qfrc_in)
print("nefc per world:", d.nefc.numpy()[:8], "| contacts:", d.nacon.numpy()[0])
print("world 0 qfrc_inverse              :", d.qfrc_inverse.numpy()[0])
print("world 0 qfrc_applied + qfrc_actuator:", qfrc_in[0])
print(f"max |qfrc_inverse - (qfrc_applied + qfrc_actuator)| over {NWORLD} worlds: {err.max():.2e}")
print(f"largest |qfrc| for scale: {np.abs(qfrc_in).max():.3f}; "
      f"largest |qfrc_constraint|: {np.abs(d.qfrc_constraint.numpy()).max():.3f}")

# world 0 in MuJoCo C: same state and qacc, then mj_inverse
mjd = mujoco.MjData(mjm)
mjw.get_data_into(mjd, mjm, d, world_id=0)
mujoco.mj_inverse(mjm, mjd)
ec = np.abs(d.qfrc_inverse.numpy()[0] - mjd.qfrc_inverse).max()
print(f"world 0 |MJWarp - mj_inverse| max: {ec:.2e} "
      f"(relative to max |qfrc_constraint|: {ec / np.abs(mjd.qfrc_constraint).max():.1e})")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

One world per time sample: a whole trajectory's torques in one call, then an open-loop replay.

In [ ]:
# Example 14.11 - ex11_inverse_trajectory.py
"""Example 14.11 - inverse dynamics over a whole trajectory, one world per time sample.

The double pendulum (M2) must follow q1 = 0.6 sin(pi t), q2 = 0.4 sin(pi t + 1) for 2 s at
dt = 0.01 (200 samples). Instead of looping over time, we put sample k in world k: qpos, qvel,
qacc of that instant, and ONE mjw.inverse call returns all 200 torque vectors in d.qfrc_inverse.
Two ways to get qvel and qacc:
  A. analytic derivatives of the sines (continuous time);
  B. finite differences that match the semi-implicit Euler update
     (qvel_k = (q_k - q_{k-1}) / dt, qacc_k = (qvel_{k+1} - qvel_k) / dt).
Then both torque tables are replayed open loop through qfrc_applied in 2 worlds (a tiny kernel
copies row k on the device each step) and the tracking error is printed.
Run: python ex11_inverse_trajectory.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

DT = 0.01
NSAMPLE = 200          # 2 s of samples = 200 worlds (small enough for a CPU run)
AMP, PHASE, W = np.array([0.6, 0.4]), np.array([0.0, 1.0]), np.pi


def traj(t):
    s = np.sin(W * t[:, None] + PHASE)
    c = np.cos(W * t[:, None] + PHASE)
    return AMP * s, AMP * W * c, -AMP * W * W * s


@wp.kernel
def set_row(table: wp.array3d(dtype=float), k: int, out: wp.array2d(dtype=float)):
    worldid, dofid = wp.tid()
    out[worldid, dofid] = table[worldid, k, dofid]


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
t = np.arange(NSAMPLE + 2) * DT
q, qd_a, qdd_a = traj(t)
# method B: velocities and accelerations consistent with semi-implicit Euler
qd_b = np.vstack([qd_a[:1], np.diff(q, axis=0) / DT])          # qvel_k = (q_k - q_{k-1}) / dt
qdd_b = np.vstack([np.diff(qd_b, axis=0) / DT, qdd_a[-1:]])    # qacc_k = (qvel_{k+1} - qvel_k)/dt

tables = {}
dinv = mjw.make_data(mjm, nworld=NSAMPLE)       # one world per time sample
for name, qd, qdd in (("A analytic", qd_a, qdd_a), ("B Euler-consistent", qd_b, qdd_b)):
    dinv.qpos.assign(q[:NSAMPLE].astype(np.float32))
    dinv.qvel.assign(qd[:NSAMPLE].astype(np.float32))
    dinv.qacc.assign(qdd[:NSAMPLE].astype(np.float32))
    mjw.inverse(m, dinv)                          # 200 time samples, one call
    tables[name] = dinv.qfrc_inverse.numpy().copy()
wp.synchronize()
tauA, tauB = tables["A analytic"], tables["B Euler-consistent"]
print(f"inverse: {NSAMPLE} worlds = {NSAMPLE} time samples, qfrc_inverse shape {tauA.shape}")
print("tau at t = 0, 0.5, 1.0 s (method A):\n", tauA[[0, 50, 100]])
diff = np.abs(tauA - tauB)
# sample 0 differs most: qd_b[0] = qd_a[0], so qdd_b[0] contains a velocity-jump kick
print(f"|tauA - tauB| at k = 0: {diff[0].max():.3f} N m; max over k >= 1: {diff[1:].max():.4f} N m")

# open-loop replay of both tables in 2 worlds at once
drep = mjw.make_data(mjm, nworld=2)
drep.qpos.assign(np.tile(q[0], (2, 1)).astype(np.float32))
drep.qvel.assign(np.stack([qd_a[0], qd_b[0]]).astype(np.float32))
table = wp.array(np.stack([tauA, tauB]).astype(np.float32), dtype=float)   # (2, NSAMPLE, nv)
err = np.zeros((NSAMPLE, 2))
for k in range(NSAMPLE):
    wp.launch(set_row, dim=(2, mjm.nv), inputs=[table, k], outputs=[drep.qfrc_applied])
    mjw.step(m, drep)                             # state k -> k+1 with torque k
    err[k] = np.abs(drep.qpos.numpy() - q[k + 1]).max(axis=1)   # demo: reads every step
for j, name in enumerate(tables):
    rms = np.sqrt((err[:, j] ** 2).mean())
    print(f"replay {name:18s}: RMS tracking error {rms:.2e} rad, final {err[-1, j]:.2e} rad")
print("peak |tau| per joint (method B):", np.abs(tauB).max(axis=0))

if args.dump_json:
    dec = slice(0, NSAMPLE, 2)
    with open(args.dump_json, "w") as f:
        json.dump({"dt": 2 * DT, "t": np.round(t[:NSAMPLE][dec], 3).tolist(),
                   "q": np.round(q[:NSAMPLE][dec], 3).tolist(),
                   "tauA": np.round(tauA[dec], 3).tolist(), "tauB": np.round(tauB[dec], 3).tolist(),
                   "errA": [float(f"{v:.3g}") for v in err[dec, 0]],
                   "errB": [float(f"{v:.3g}") for v in err[dec, 1]]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Plot Example 14.11: torques of both methods and the open-loop tracking error (log scale).
# Run the Example 14.11 cell first (it defines t, tauA, tauB, err, NSAMPLE).
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ts = t[:NSAMPLE]
    for j, (ca, cb) in enumerate([("C0", "C2"), ("C1", "C3")]):
        ax[0].plot(ts, tauA[:, j], ca, label=f"tau{j + 1} analytic (A)")
        ax[0].plot(ts, tauB[:, j], cb + "--", label=f"tau{j + 1} Euler-consistent (B)")
    ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("N m"); ax[0].legend(fontsize=8)
    ax[1].semilogy(ts, err[:, 0] + 1e-12, label="replay A")
    ax[1].semilogy(ts, err[:, 1] + 1e-12, label="replay B")
    ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("max |q - q_ref| [rad]"); ax[1].legend()
    plt.tight_layout(); plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 8. Part III checkpoint: inverse-dynamics trajectory tracking for the humanoid arm
Torso bolted, Euler implicit damping on, right arm on a smooth trajectory. Compare plain `inverse` with `EnableBit.INVDISCRETE`.

In [ ]:
# Example 14.12 - ex12_checkpoint_humanoid_arm.py
"""Example 14.12 - Part III checkpoint: inverse-dynamics trajectory tracking for the humanoid arm.

The shipped humanoid (M8) with its torso bolted in the air (MjSpec removes the free joint) and
Euler's implicit joint damping switched back on (the shipped file disables eulerdamp). The right
shoulder and elbow follow smooth sines for 2 s; every other joint holds 0.
  1. Build Euler-consistent qpos/qvel/qacc samples (finite differences, Example 14.11).
  2. Put sample k in world k and call mjw.inverse once: 400 torque vectors.
     Do it twice: plain inverse, and with EnableBit.INVDISCRETE, which first converts the
     discrete-time qacc to a continuous one (inverse.py::discrete_acc: (M + h D) qacc for Euler).
  3. Replay both torque tables open loop through qfrc_applied, with gravity, in 2 worlds.
Prints the RMS tracking error of the arm joints, the peak torque per arm joint and whether the
humanoid's own arm motors (gear 20, 20, 40; ctrlrange [-1, 1]) could deliver it.
Run: python ex12_checkpoint_humanoid_arm.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
spec = mujoco.MjSpec.from_file(str(HUMANOID_PATH))
spec.delete(spec.joint("root"))                # bolt the torso to the world ...
for key in list(spec.keys):                    # ... (keyframes still have the 28-wide qpos)
    spec.delete(key)
spec.body("torso").pos = [0, 0, 1.5]           # feet clear of the floor
spec.option.disableflags &= ~int(mujoco.mjtDisableBit.mjDSBL_EULERDAMP)   # implicit damping on
mjm = spec.compile()
m = mjw.put_model(mjm)

ARM = ["shoulder1_right", "shoulder2_right", "elbow_right"]
arm_dof = [int(mjm.jnt_dofadr[mjm.joint(n).id]) for n in ARM]
arm_act = [mjm.actuator(n).id for n in ARM]
DT = float(mjm.opt.timestep)                   # 0.005 s
NSAMPLE = 400                                  # 2 s -> 400 worlds for one inverse call

t = np.arange(NSAMPLE + 2) * DT
q = np.zeros((t.size, mjm.nq))
q[:, arm_dof[0]] = 0.5 * np.sin(np.pi * t)
q[:, arm_dof[1]] = 0.4 * np.sin(np.pi * t + 1.0)
q[:, arm_dof[2]] = -0.6 + 0.5 * np.sin(np.pi * t + 2.0)
qd = np.vstack([np.zeros((1, mjm.nv)), np.diff(q, axis=0) / DT])     # Euler-consistent
qd[0] = qd[1]
qdd = np.vstack([np.diff(qd, axis=0) / DT, np.zeros((1, mjm.nv))])

dinv = mjw.make_data(mjm, nworld=NSAMPLE, nconmax=8, njmax=64)
tables = {}
for name, flag in (("plain inverse", 0), ("INVDISCRETE", int(mjw.EnableBit.INVDISCRETE))):
    m.opt.enableflags = flag                   # a Python int, read when inverse() is called
    dinv.qpos.assign(q[:NSAMPLE].astype(np.float32))
    dinv.qvel.assign(qd[:NSAMPLE].astype(np.float32))
    dinv.qacc.assign(qdd[:NSAMPLE].astype(np.float32))
    mjw.inverse(m, dinv)
    tables[name] = dinv.qfrc_inverse.numpy().copy()
m.opt.enableflags = 0
wp.synchronize()
print(f"humanoid, torso fixed: nq={mjm.nq} nv={mjm.nv}; inverse over {NSAMPLE} worlds "
      f"(= {NSAMPLE * DT:.1f} s of samples)")


@wp.kernel
def set_row(table: wp.array3d(dtype=float), k: int, out: wp.array2d(dtype=float)):
    worldid, dofid = wp.tid()
    out[worldid, dofid] = table[worldid, k, dofid]


drep = mjw.make_data(mjm, nworld=2, nconmax=8, njmax=64)
drep.qpos.assign(np.tile(q[0], (2, 1)).astype(np.float32))
drep.qvel.assign(np.tile(qd[0], (2, 1)).astype(np.float32))
table = wp.array(np.stack(list(tables.values())).astype(np.float32), dtype=float)
err = np.zeros((NSAMPLE, 2, len(ARM)))
for k in range(NSAMPLE):
    wp.launch(set_row, dim=(2, mjm.nv), inputs=[table, k], outputs=[drep.qfrc_applied])
    mjw.step(m, drep)
    err[k] = drep.qpos.numpy()[:, arm_dof] - q[k + 1, arm_dof]      # demo: reads every step

for j, name in enumerate(tables):
    rms = np.sqrt((err[:, j] ** 2).mean(axis=0))
    print(f"replay with {name:13s}: RMS arm error [rad] "
          + " ".join(f"{v:.1e}" for v in rms) + f"  max {np.abs(err[:, j]).max():.1e}")
tau = tables["INVDISCRETE"]
peak = np.abs(tau[:, arm_dof]).max(axis=0)
gear = mjm.actuator_gear[arm_act, 0]
for n, p, g in zip(ARM, peak, gear):
    print(f"  {n:16s} peak |tau| = {p:6.3f} N m  -> ctrl = tau / gear = {p / g:.3f} "
          f"(gear {g:.0f}, ctrlrange [-1, 1])")
print("non-arm joints: peak holding torque", f"{np.abs(np.delete(tau, arm_dof, 1)).max():.2f} N m")
print("overflow:", "none" if not drep.overflow.numpy().any() else drep.overflow.numpy())

if args.dump_json:
    dec = slice(0, NSAMPLE, 5)
    with open(args.dump_json, "w") as f:
        json.dump({"joints": ARM, "t": np.round(t[:NSAMPLE][dec], 3).tolist(),
                   "q": np.round(q[:NSAMPLE][dec][:, arm_dof], 3).tolist(),
                   "tau": np.round(tau[dec][:, arm_dof], 3).tolist(),
                   "err_plain": [[float(f"{v:.3g}") for v in r] for r in err[dec, 0]],
                   "err_inv": [[float(f"{v:.3g}") for v in r] for r in err[dec, 1]]},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Plot Example 14.12: arm torques and the tracking error with and without INVDISCRETE.
# Run the Example 14.12 cell first (it defines t, tau, err, arm_dof, ARM, NSAMPLE).
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ts = t[:NSAMPLE]
    for k, name in enumerate(ARM):
        ax[0].plot(ts, tau[:, arm_dof[k]], label=name)
    ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("required torque [N m]"); ax[0].legend(fontsize=8)
    ax[1].semilogy(ts, np.abs(err[:, 0]).max(axis=1) + 1e-12, label="plain inverse")
    ax[1].semilogy(ts, np.abs(err[:, 1]).max(axis=1) + 1e-12, label="INVDISCRETE")
    ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("max arm error [rad]"); ax[1].legend()
    plt.tight_layout(); plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## Exercise 14.1: decode a framequat sensor
Slice the quaternion out of `d.sensordata` and assert that it has unit norm in every world.

In [ ]:
# Exercise 14.1: decode a framequat sensor and assert unit norm in every world.
FRAMEQUAT_XML = """
<mujoco>
  <worldbody><body name="box" pos="0 0 1"><freejoint/>
    <geom type="box" size=".1 .2 .3" mass="1"/><site name="s"/></body></worldbody>
  <sensor><framequat name="quat" objtype="site" objname="s"/></sensor>
</mujoco>"""
mjm = mujoco.MjModel.from_xml_string(FRAMEQUAT_XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4)
qvel = np.zeros((4, 6), np.float32)
qvel[:, 3:] = np.random.default_rng(0).normal(size=(4, 3))   # tumble: random angular velocity
d.qvel.assign(qvel)
for _ in range(100):
    mjw.step(m, d)
mjw.forward(m, d)
# your code: find adr and dim of sensor "quat", slice d.sensordata, compute the norms
quat = ...
norm = ...
assert quat.shape == (4, 4)
assert np.allclose(norm, 1.0, atol=1e-5)
print("exercise 14.1 passed")

In [ ]:
#@title Exercise 14.1: solution (double-click to show the code)

FRAMEQUAT_XML = """
<mujoco>
  <worldbody><body name="box" pos="0 0 1"><freejoint/>
    <geom type="box" size=".1 .2 .3" mass="1"/><site name="s"/></body></worldbody>
  <sensor><framequat name="quat" objtype="site" objname="s"/></sensor>
</mujoco>"""
mjm = mujoco.MjModel.from_xml_string(FRAMEQUAT_XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4)
qvel = np.zeros((4, 6), np.float32)
qvel[:, 3:] = np.random.default_rng(0).normal(size=(4, 3))
d.qvel.assign(qvel)
for _ in range(100):
    mjw.step(m, d)
mjw.forward(m, d)
sid = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SENSOR, "quat")
adr, dim = int(mjm.sensor_adr[sid]), int(mjm.sensor_dim[sid])
quat = d.sensordata.numpy()[:, adr:adr + dim]          # (w, x, y, z) per world
norm = np.linalg.norm(quat, axis=1)
assert quat.shape == (4, 4)
assert np.allclose(norm, 1.0, atol=1e-5)
# the site has no offset, so the sensor equals the body quaternion up to sign
assert np.allclose(np.abs(np.sum(quat * d.xquat.numpy()[:, 1], axis=1)), 1.0, atol=1e-5)
print("exercise 14.1 passed: max |norm - 1| < 1e-5:", bool(np.abs(norm - 1).max() < 1e-5))

## Exercise 14.2: the delay in steps
Assert that the motor of Example 14.8 starts pushing exactly `round(delay / timestep)` steps late.

In [ ]:
# Exercise 14.2: measure the actuator delay of Example 14.8 in steps and assert it.
DELAY_XML = """
<mujoco>
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody><body><joint name="slide" type="slide" axis="1 0 0"/>
    <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/></body></worldbody>
  <actuator><motor joint="slide" delay="0.03" nsample="5"/></actuator>
</mujoco>"""
# your code: step with ctrl = 1, record actuator_force, find the first non-zero step,
# and compare it with round(actuator_delay / timestep)
nzero = None  # your code
expected = None  # your code
assert nzero is not None and expected is not None, "replace the placeholders"
assert nzero == expected, (nzero, expected)
print("exercise 14.2 passed")

In [ ]:
#@title Exercise 14.2: solution (double-click to show the code)

DELAY_XML = """
<mujoco>
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody><body><joint name="slide" type="slide" axis="1 0 0"/>
    <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/></body></worldbody>
  <actuator><motor joint="slide" delay="0.03" nsample="5"/></actuator>
</mujoco>"""
mjm = mujoco.MjModel.from_xml_string(DELAY_XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)
d.ctrl.fill_(1.0)
force = []
for _ in range(8):
    mjw.step(m, d)
    force.append(float(d.actuator_force.numpy()[0, 0]))
nzero = next(i for i, f in enumerate(force) if f != 0.0)
expected = round(float(mjm.actuator_delay[0]) / mjm.opt.timestep)
assert nzero == expected, (nzero, expected)
print("exercise 14.2 passed: delay =", nzero, "steps;", force)

## Stretch: when does delay destabilize a PD controller?
`actuator_delay` is a batched model field, so every world can have its own delay. The starter below sweeps 0-0.11 s; find the boundary, then explore.

In [ ]:
# Stretch: a delayed PD controller. One world per actuator delay (actuator_delay is a batched field).
PD_XML = """
<mujoco>
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody><body><joint name="slide" type="slide" axis="1 0 0"/>
    <geom type="box" size=".1 .1 .1" mass="1" contype="0" conaffinity="0"/></body></worldbody>
  <actuator><motor joint="slide" delay="0.01" nsample="20"/></actuator>
</mujoco>"""
NW = 12
mjm = mujoco.MjModel.from_xml_string(PD_XML)
m = mjw.put_model(mjm, batch_sizes={"actuator_delay": NW})
delays = np.arange(NW) * 0.01
m.actuator_delay.assign(delays[:, None].astype(np.float32))
d = mjw.make_data(mjm, nworld=NW)
d.qpos.fill_(1.0)


@wp.kernel
def pd(qpos: wp.array2d(dtype=float), qvel: wp.array2d(dtype=float), kp: float, kd: float,
       ctrl: wp.array2d(dtype=float)):
    w = wp.tid()
    ctrl[w, 0] = -kp * qpos[w, 0] - kd * qvel[w, 0]


for i in range(800):
    wp.launch(pd, dim=NW, inputs=[d.qpos, d.qvel, 100.0, 5.0], outputs=[d.ctrl])
    mjw.step(m, d)
for dl, a in zip(delays, np.abs(d.qpos.numpy()[:, 0])):
    print(f"delay {dl:.2f} s  |q| after 8 s {a:.3e}")
# Now: refine the delay grid between the last stable and the first unstable world (on a GPU use
# thousands of worlds), vary kd, and compare with the phase-margin estimate of a delayed PD loop.

## Part III review
1. Which stage computes a touch sensor, and why? (acc: it sums `efc.force`.)
2. Which call fills `qLD` and what goes wrong if you solve with an old one? (Ch 11)
3. Where does `rne_postconstraint` run, and which sensors make `forward` call it? (Ch 12, §3 here)
4. What does the implicitfast integrator drop compared with implicit? (Ch 13)
5. Under RK4, from which moment of the step do sensors come? (the start)
6. Why does inverse dynamics with INVDISCRETE reproduce an Euler trajectory exactly, while plain inverse does not when joint damping is integrated implicitly?